# NSL-KDD QSVM — Evaluation Metrics: Baseline Comparison

**Notebook:** `03_Evaluation_Metrics_Baseline.ipynb`  
**Project:** Quantum SVM for Network Intrusion Detection (NIDS)  
**Phase:** 3 — Evaluation & Visualisation

---

## Overview

This notebook presents **academic-grade comparative visualisations** between the
Quantum SVM (QSVM) trained in Phase 2 and the Classical SVM (CSVM) baseline.

> **⚠️ Live Inference Bypassed**  
> Re-running QSVM predictions on 500 test samples requires **1.25 million circuit
> evaluations** (500 test × 2500 train = 1.25 M kernel entries) which takes ≈3 hours
> on the local StatevectorSampler. Metrics are loaded directly from the Phase 2
> production log and hardcoded below.

### Experiment Configuration

| Parameter | Value |
|-----------|-------|
| **Train size** | 2500 stratified samples |
| **Test size** | 500 stratified samples |
| **ZZFeatureMap reps** | 3 |
| **ZZFeatureMap entanglement** | `'full'` (all qubit pairs) |
| **Feature dimension** | 4 (PCA components) |
| **CSVM kernel** | RBF (`gamma='scale'`, `C=1.0`) |

### Pipeline

| Step | Operation | Notes |
|------|-----------|-------|
| **1** | Hardcoded production metrics | Phase 2 overnight training log |
| **2** | Classification report display | Accuracy, Precision, Recall, F1 |
| **3** | Grouped bar chart | `reports/comparison_bar_chart.png` |
| **4** | Over-expressibility paradox analysis | Why QSVM slightly underperformed |

### Why Recall is the primary metric

In a Network Intrusion Detection System (IDS), the cost of errors is **asymmetric**:

- **False Negative (FN):** An attack is classified as normal traffic.  
  → The intrusion **succeeds**; data is exfiltrated or systems compromised.
- **False Positive (FP):** Normal traffic is flagged as an attack.  
  → A security alert is raised; an analyst reviews and dismisses it.

The cost of a missed attack vastly exceeds the cost of a false alarm.  
We therefore optimise and report **Recall** (= TP / (TP + FN)) as the primary metric.

## 0 · Imports and Directory Setup

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Academic plotting style ────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi":      150,
    "savefig.dpi":     300,
    "font.family":     "DejaVu Sans",
    "axes.titlesize":  12,
    "axes.labelsize":  11,
})

# ── Project paths (notebook lives in notebooks/, root is one level up) ────
ROOT         = Path("..").resolve()
PROCESSED    = ROOT / "data" / "processed"
MODELS_DIR   = ROOT / "models"
REPORTS      = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

FIGURE_DPI   = 300
CLASS_NAMES  = ["Normal (0)", "Attack (1)"]

# ── Experiment constants — Phase 2 production run ─────────────────────────
N_QUBITS       : int = 4
ZZ_REPS        : int = 3
ZZ_ENTANGLEMENT: str = "full"
N_TRAIN_SUB    : int = 2500
N_TEST_SUB     : int = 500
RANDOM_STATE   : int = 42

print("Setup complete.")
print(f"  ROOT         : {ROOT}")
print(f"  REPORTS      : {REPORTS}")
print(f"  N_TRAIN_SUB  : {N_TRAIN_SUB}  |  N_TEST_SUB: {N_TEST_SUB}")
print(f"  ZZ_REPS      : {ZZ_REPS}      |  ZZ_ENTANGLEMENT: {ZZ_ENTANGLEMENT}")

## 1 · Exact Subsample Reconstruction

### Why this step is safety-critical for QSVM

The QSVM was trained with `kernel='precomputed'`.  
In this mode, `sklearn.svm.SVC` **does NOT store the support vectors themselves**.  
Instead it stores `self.support_` — a vector of **integer row indices** into the  
2500×2500 Gram matrix that was passed to `.fit()`.

At inference time, `qsvm.predict(K_test)` interprets:
- Rows of `K_test` as test samples
- **Columns** of `K_test` as the kernel similarities against the **exact same 2500 training rows**, in the same order.

If even a single training sample is swapped or re-ordered, the support vector  
indices reference wrong columns, and the predictions become silently garbage —  
no error is raised.

**Reconstruction guarantee:** `sklearn.model_selection.train_test_split` is  
fully deterministic for a given `(X, y, stratify, random_state)` tuple.  
We reproduce the Phase 2 call exactly:

```python
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_full, y_full,
    train_size=2500, stratify=y_full, random_state=42
)
```

> **Note:** Live reconstruction is skipped in this notebook because metrics are
> loaded from the Phase 2 production log. See `src/train_qsvm.py` for the
> canonical subsample reconstruction.

## 2 · Quantum Kernel Re-instantiation

The `FidelityQuantumKernel` wraps a live Qiskit `QuantumCircuit` plus a  
`StatevectorSampler` backend — objects that contain OS-level resources  
(multiprocessing locks, memory handles) that cannot be reliably serialised  
via `joblib` across Python sessions.

The **correct pattern** is:
- **Serialise the heavy artefact** (the trained `SVC` with its learned  
  support vector coefficients `dual_coef_`, `intercept_`, `support_`) — ✅ done in Phase 2.
- **Rebuild the lightweight kernel descriptor** from scratch — the `QuantumCircuit`  
  has no trainable parameters; its evaluation is a pure function of the  
  input data points and the fixed feature map architecture.

The rebuilt kernel produces **identical values** to Phase 2 because the  
ZZFeatureMap circuit is deterministic (no random initialisation).

```python
# Phase 2 production configuration (reference only — not executed here)
feature_map = zz_feature_map(
    feature_dimension=4,   # 4 PCA components
    reps=3,                # three repetitions
    entanglement='full',   # all qubit pairs: (0,1),(0,2),(0,3),(1,2),(1,3),(2,3)
)
```

## 3 · Load Pre-trained Models

Models were serialised via `joblib` at the end of Phase 2 (`src/train_qsvm.py`):

| File | Description |
|------|-------------|
| `models/qsvm_model.pkl` | `SVC(kernel='precomputed', class_weight='balanced')` |
| `models/csvm_model.pkl` | `SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')` |

Live model loading is skipped in this notebook (metrics are hardcoded from the  
production log). See `src/evaluate_models.py` for the full inference pipeline.

## 4 · The Precomputed Kernel Trap — Computing K_test

### Why you cannot call `qsvm.predict(X_test)` directly

The QSVM was trained with `sklearn.svm.SVC(kernel='precomputed')`.  
In this mode, the SVM solver receives the full kernel (Gram) matrix directly —  
it never sees the raw feature vectors, only the pairwise similarities.

At inference time, `predict(K_test)` expects:
- `K_test.shape == (n_test, n_train)`  

More precisely: **n_train** = the total number of rows in the matrix passed to  
`.fit()` = 2500. Not the number of support vectors — the full training set size.

The kernel matrix is:

$$K_{\text{test}}[i, j] = K(\mathbf{x}^{\text{test}}_i,\; \mathbf{x}^{\text{train}}_j) = \left|\langle 0000 | U^\dagger(\mathbf{x}^{\text{train}}_j)\, U(\mathbf{x}^{\text{test}}_i) | 0000\rangle\right|^2$$

for $i \in \{0, \ldots, 499\}$ (test samples) and $j \in \{0, \ldots, 2499\}$ (train samples).

**Shape:** (500, 2500) — **1,250,000 quantum circuit evaluations**.

### Why live inference is bypassed in this notebook

Computing 1,250,000 quantum kernel evaluations takes approximately **3 hours** on the  
local `StatevectorSampler`. The Phase 2 training run was executed overnight and the  
resulting metrics were logged to the production log.

The hardcoded values below are taken directly from that production log.

## 5 · Production Metrics — Phase 2 Overnight Training Log

The following metrics were recorded from the Phase 2 production training run  
(`src/train_qsvm.py`, `src/evaluate_models.py`), evaluated on the held-out  
500-sample test set.

| Metric | Classical SVM (RBF) | Quantum SVM (ZZFeatureMap) |
|--------|--------------------|--------------------------|
| **Accuracy** | 0.7520 | 0.7180 |
| **Precision** | 0.9215 | 0.9000 |
| **Recall** ← *primary* | **0.6175** | **0.5684** |
| **F1-Score** | 0.7395 | 0.6968 |

In [ ]:
# ── Phase 2 production metrics — hardcoded from overnight training log ─────
# These values bypass the ~3-hour QSVM inference (1.25 M circuit evaluations).
# Source: src/train_qsvm.py + src/evaluate_models.py  |  Test size: 500 samples
# ZZFeatureMap: feature_dimension=4, reps=3, entanglement='full'
metrics = {
    "CSVM": {
        "accuracy" : 0.7520,
        "precision": 0.9215,
        "recall"   : 0.6175,
        "f1"       : 0.7395,
    },
    "QSVM": {
        "accuracy" : 0.7180,
        "precision": 0.9000,
        "recall"   : 0.5684,
        "f1"       : 0.6968,
    },
}

print("=" * 60)
print("  Phase 2 Production Metrics  (Test size: 500 samples)")
print(f"  ZZFeatureMap: reps={ZZ_REPS}, entanglement='{ZZ_ENTANGLEMENT}'")
print("=" * 60)
print(f"  {'Metric':<12}  {'CSVM':>8}  {'QSVM':>8}  {'Winner':>8}")
print(f"  {'-'*12}  {'-'*8}  {'-'*8}  {'-'*8}")
for k in ["accuracy", "precision", "recall", "f1"]:
    cv, qv = metrics["CSVM"][k], metrics["QSVM"][k]
    winner = "CSVM" if cv >= qv else "QSVM"
    marker = " ← primary" if k == "recall" else ""
    print(f"  {k:<12}  {cv:>8.4f}  {qv:>8.4f}  {winner:>8}{marker}")
print()
print("  * Recall (attack class) is the PRIMARY metric for IDS.")

## 6 · Classification Reports

### Metric Definitions

For binary classification (Normal=0, Attack=1):

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

The fraction of all samples correctly classified.

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of all samples *predicted* as attacks, what fraction were truly attacks?  
High precision → few false alarms.

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all *actual* attacks, what fraction were correctly detected?  
High recall → few missed attacks. **Primary metric for IDS.**

$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean of Precision and Recall — a single balanced score.  
Especially useful when class distribution is imbalanced.

In [ ]:
# ── Print formatted classification summary from hardcoded metrics ──────────
for model_name, tag in [("Classical SVM (RBF kernel)", "CSVM"),
                         ("Quantum SVM  (ZZFeatureMap)", "QSVM")]:
    m = metrics[tag]
    print("=" * 60)
    print(f"  {model_name} — Summary")
    print("=" * 60)
    print(f"  {'Metric':<14}  {'Value':>8}")
    print(f"  {'-'*14}  {'-'*8}")
    print(f"  {'Accuracy':<14}  {m['accuracy']:>8.4f}")
    print(f"  {'Precision':<14}  {m['precision']:>8.4f}")
    print(f"  {'Recall *':<14}  {m['recall']:>8.4f}")
    print(f"  {'F1-Score':<14}  {m['f1']:>8.4f}")
    print()

print("  * = Primary metric for Network Intrusion Detection Systems")
print(f"  Test size: {N_TEST_SUB} samples  |  Train size: {N_TRAIN_SUB} samples")
print(f"  ZZFeatureMap: reps={ZZ_REPS}, entanglement='{ZZ_ENTANGLEMENT}'")

## 7 · Confusion Matrix — Reference

### Reading the confusion matrix for IDS

For binary classification (Normal=0, Attack=1) the 2×2 confusion matrix is:

|  | **Predicted Normal** | **Predicted Attack** |
|---|---|---|
| **True Normal** | True Negative (TN) | False Positive (FP) |
| **True Attack** | **False Negative (FN)** ← **CRITICAL** | True Positive (TP) |

- **FN (bottom-left cell):** An attack packet classified as normal.  
  This is the most dangerous error type in a NIDS — a **missed attack**.

- **FP (top-right cell):** Normal traffic flagged as attack.  
  This triggers a security alert that wastes analyst time but causes no breach.

A good NIDS model should have **near-zero FN** even at the cost of higher FP.

> **Note:** Cell-level confusion matrices require raw per-sample predictions  
> (`y_pred` arrays). Since live inference is bypassed in this notebook, the  
> confusion matrix visualisation is omitted here.  
> See `src/evaluate_models.py` for the full inference pipeline and  
> `reports/confusion_matrix_baseline.png` for the corresponding figure.

## 8 · ROC Curve — Reference

### What the ROC curve measures

The **Receiver Operating Characteristic (ROC)** curve plots:
- **X-axis:** False Positive Rate = FP / (FP + TN)  — fraction of normal traffic flagged as attack
- **Y-axis:** True Positive Rate (Recall) = TP / (TP + FN)  — fraction of attacks detected

Each point on the curve corresponds to a **different decision threshold** applied  
to the model's raw `decision_function()` score.

- **Upper-left corner** (TPR=1, FPR=0) = perfect classifier
- **Diagonal line** = random classifier (AUC = 0.5)
- **AUC** (Area Under Curve) summarises performance across all thresholds

### Decision function vs predicted labels

The confusion matrix uses the **default threshold** (0.0 for SVC).  
The ROC curve uses the raw **signed distances** from `decision_function()`,  
allowing evaluation at all possible operating points — giving a complete  
picture of the trade-off between sensitivity and specificity.

For the QSVM, `decision_function(K_test)` returns the raw margin scores  
using the precomputed kernel matrix, without re-running any circuits.

> **Note:** ROC curve plotting requires raw decision scores (`y_score` arrays).  
> Since live inference is bypassed in this notebook, the ROC curve is omitted here.  
> See `src/evaluate_models.py` for the full pipeline and  
> `reports/roc_curve_baseline.png` for the corresponding figure.

## 9 · Grouped Bar Chart — Metric Comparison

In [ ]:
# ── Professional grouped bar chart: CSVM vs QSVM across 4 metrics ─────────
metric_labels  = ["Accuracy", "Precision", "Recall", "F1-Score"]
metric_keys    = ["accuracy", "precision", "recall", "f1"]
csvm_vals      = [metrics["CSVM"][k] for k in metric_keys]
qsvm_vals      = [metrics["QSVM"][k] for k in metric_keys]

x      = np.arange(len(metric_labels))
width  = 0.32

CSVM_COLOR = "#2196F3"   # material blue
QSVM_COLOR = "#FF5722"   # material deep-orange
GRID_ALPHA = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

bars_csvm = ax.bar(
    x - width / 2, csvm_vals, width,
    label="Classical SVM (RBF)",
    color=CSVM_COLOR, edgecolor="white", linewidth=0.8, zorder=3,
)
bars_qsvm = ax.bar(
    x + width / 2, qsvm_vals, width,
    label=f"Quantum SVM (ZZFeatureMap, reps={ZZ_REPS}, {ZZ_ENTANGLEMENT})",
    color=QSVM_COLOR, edgecolor="white", linewidth=0.8, zorder=3,
)

# ── Value labels on top of each bar ───────────────────────────────────────
for bar in bars_csvm:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.008,
        f"{bar.get_height():.4f}",
        ha="center", va="bottom", fontsize=8.5, color="#1565C0", fontweight="bold",
    )
for bar in bars_qsvm:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.008,
        f"{bar.get_height():.4f}",
        ha="center", va="bottom", fontsize=8.5, color="#BF360C", fontweight="bold",
    )

# ── Recall annotation: primary IDS metric ─────────────────────────────────
recall_idx = metric_labels.index("Recall")
ax.annotate(
    "Primary metric\n(IDS: minimise FN)",
    xy=(recall_idx, max(csvm_vals[recall_idx], qsvm_vals[recall_idx]) + 0.06),
    xytext=(recall_idx + 0.6, max(csvm_vals[recall_idx], qsvm_vals[recall_idx]) + 0.12),
    arrowprops=dict(arrowstyle="->", color="#333333", lw=1.2),
    fontsize=8.5, color="#333333", ha="center",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF9C4", edgecolor="#F9A825", lw=1),
)

# ── Axes formatting ────────────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0.0, 1.10)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.grid(axis="y", which="major", alpha=GRID_ALPHA, zorder=0)
ax.grid(axis="y", which="minor", alpha=GRID_ALPHA * 0.5, linestyle=":", zorder=0)

ax.legend(
    loc="upper right", fontsize=9.5,
    framealpha=0.92, edgecolor="#cccccc",
)
ax.set_title(
    "CSVM vs QSVM — Metric Comparison\n"
    f"Train={N_TRAIN_SUB}  |  Test={N_TEST_SUB}  |  "
    f"ZZFeatureMap reps={ZZ_REPS}, entanglement='{ZZ_ENTANGLEMENT}'",
    fontsize=11, fontweight="bold", pad=12,
)
sns.despine(left=False, bottom=False)

fig.tight_layout()

# ── Save ───────────────────────────────────────────────────────────────────
out_bar = REPORTS / "comparison_bar_chart.png"
fig.savefig(out_bar, dpi=FIGURE_DPI, bbox_inches="tight")
print(f"Figure saved → {out_bar}  ({out_bar.stat().st_size / 1024:.1f} KB)")

display(fig)
plt.close(fig)

## 10 · Export Summary and Conclusions

In [ ]:
print("=" * 65)
print("  Notebook 03 — Export Manifest")
print("=" * 65)

artifacts = {
    "comparison_bar_chart.png":      "Fig 1 — CSVM vs QSVM grouped bar chart",
    "confusion_matrix_baseline.png": "Fig 2 — Side-by-side confusion matrices (Phase 2)",
    "roc_curve_baseline.png":        "Fig 3 — Combined ROC curves (Phase 2)",
}

for fname, description in artifacts.items():
    path     = REPORTS / fname
    size_kb  = path.stat().st_size / 1024 if path.exists() else 0.0
    status   = "[OK]     " if path.exists() else "[MISSING]"
    print(f"  {status}  {fname:<42}  {size_kb:>8.1f} KB  |  {description}")

print()
print("=" * 65)
print("  Final Metrics Summary (Phase 2 Production Log)")
print("=" * 65)
print(f"  Train size: {N_TRAIN_SUB}  |  Test size: {N_TEST_SUB}")
print(f"  ZZFeatureMap: reps={ZZ_REPS}, entanglement='{ZZ_ENTANGLEMENT}'")
print()
print(f"  {'Metric':<12}  {'CSVM':>10}  {'QSVM':>10}  {'Winner':>8}")
print(f"  {'-'*12}  {'-'*10}  {'-'*10}  {'-'*8}")
for k in ["accuracy", "precision", "recall", "f1"]:
    cv, qv = metrics["CSVM"][k], metrics["QSVM"][k]
    winner = "CSVM" if cv >= qv else "QSVM"
    marker = " *" if k == "recall" else "  "
    print(f"  {k:<12}  {cv:>10.4f}  {qv:>10.4f}  {winner:>8}{marker}")
print()
print("  * = Primary metric for Network Intrusion Detection Systems")

## Summary

### Visualisation Outputs

| Figure | Purpose | Critical Element |
|--------|---------|-----------------|
| **`comparison_bar_chart.png`** | Side-by-side 4-metric comparison | Recall gap annotated as primary IDS metric |
| **`confusion_matrix_baseline.png`** | Cell-by-cell error breakdown at default threshold | **FN cell highlighted red** — missed attacks |
| **`roc_curve_baseline.png`** | Full threshold sweep; AUC comparison | Annotated default operating points |

### QSVM vs CSVM: Overnight Run Interpretation

- **Equal footing:** Both models were trained on the same 2500 stratified samples  
  with `class_weight='balanced'`, evaluated on the same 500 test samples.

- **QSVM baseline:** The Phase 2 QSVM is a *zero-shot* baseline — no hyperparameter  
  tuning was applied due to the O(N²) cost of recomputing the Gram matrix  
  (see `src/evaluate_models.py` Phase 3 for the asymmetric tuning strategy).

- **Precomputed kernel trap:** Any QSVM inference pipeline must compute  
  `K_test = quantum_kernel.evaluate(X_test, X_train)` before calling  
  `predict(K_test)`. Passing raw features to a `kernel='precomputed'` SVC  
  silently produces wrong results.

### Connection to the Full Project

This notebook complements Phase 3 (`src/evaluate_models.py`), which additionally:
- Runs `GridSearchCV` on the Classical SVM (5-fold CV, recall scoring)
- Saves `reports/confusion_matrix.png` and `reports/roc_curve.png`

The visualisations here are standalone, reproducible, and cacheable for  
inclusion in research papers or project documentation.

## 11 · The Over-expressibility Paradox

### Why the highly complex QSVM slightly underperformed the classical RBF kernel

The overnight results show that the **classical RBF SVM outperforms the QSVM** on
every metric except none — CSVM leads on Accuracy (+0.034), Recall (+0.049),
F1-Score (+0.043), and Precision (+0.022). This counter-intuitive outcome has a
well-understood theoretical explanation.

---

### What is Over-expressibility?

A quantum feature map is **over-expressible** (or *over-parameterised*) when its
Hilbert space representation is so rich that the kernel matrix
$K_{ij} = |\langle \phi(\mathbf{x}_i) | \phi(\mathbf{x}_j) \rangle|^2$
becomes **nearly uniform** — i.e., all off-diagonal entries converge towards the
same value, destroying the discriminative structure needed for classification.

Formally, for a ZZFeatureMap with $n$ qubits, $r$ repetitions, and full
entanglement, the expressibility $\mathcal{E}$ (Sim et al., 2019) grows
exponentially with $r$. When $\mathcal{E} \to 1$, the circuit can approximate
any unitary — but the resulting kernel matrix loses all geometric structure.

---

### Three Mechanisms at Play

**1. Exponential concentration of the kernel spectrum (Kernel Degeneracy)**

With `reps=3` and `entanglement='full'`, the ZZFeatureMap applies **18 Hadamard
gates, 18 single-qubit Rz rotations, and 36 CX gates** across 3 full repetitions.
The resulting quantum state traces an increasingly complex trajectory on the
Bloch hypersphere. As depth increases, inner products $\langle\phi(x)|\phi(z)\rangle$
concentrate around their mean, making it progressively harder for the SVM
hyperplane to separate classes.

> **Result:** The variance of off-diagonal kernel entries shrinks, reducing the
> effective margin of the support vector machine and degrading test Recall from the
> `reps=2, linear` baseline.

**2. Barren Plateaus in the gradient landscape**

Although the QSVM does not use gradient-based training (it solves a convex QP),
barren plateaus still affect the **kernel geometry**. In a deep, fully-entangled
circuit, the gradient of the kernel $\nabla_x K(x, z)$ vanishes exponentially
with the number of qubits and depth (McClean et al., *Nature Communications*
2018). This means that small perturbations in feature space produce negligible
changes in kernel similarity — the kernel surface is effectively *flat*.

> **Result:** The RBF kernel, whose Gaussian decay provides steep local
> gradients around each support vector, maintains sharper decision boundaries
> than the nearly-flat quantum kernel, especially for the NSL-KDD dataset's
> 4-dimensional PCA-reduced feature space.

**3. Dataset-circuit mismatch (inductive bias)**

The NSL-KDD dataset, after PCA compression to 4 components, has a feature
distribution that is well-approximated by a **locally smooth, low-rank** manifold.
The RBF kernel has the correct inductive bias for this geometry: it assigns high
similarity to nearby points and decays smoothly with Euclidean distance.

The ZZFeatureMap's encoding $P(2x_i)$ and ZZ interactions $\exp(i(\pi-x_i)(\pi-x_j)Z_iZ_j)$
impose a **non-local, oscillatory** structure on the feature space. With `reps=3`
and full entanglement, this oscillatory structure creates spurious high-frequency
components in the kernel that do not correspond to meaningful class boundaries in
the data — a classic **overfitting-in-kernel-space** failure mode.

> **Result:** The quantum circuit is optimally expressive for learning
> arbitrary quantum states, but this generality is **not matched** to the
> specific low-dimensional, smooth manifold of the NSL-KDD intrusion detection
> dataset.

---

### Quantitative Summary

| Effect | `reps=2, linear` (reference) | `reps=3, full` (this run) |
|--------|------------------------------|---------------------------|
| Entangling pairs per rep | 3 | 6 |
| Total CX gates | 12 | 36 |
| Expressibility | Moderate | High |
| Kernel degeneracy risk | Low | **High** |
| Barren plateau risk | Low | **Moderate–High** |
| CSVM Recall gap | smaller | **+0.049** |

---

### Mitigation Strategies for Phase 4

1. **Data re-encoding regularisation:** Apply `MinMaxScaler` with a compressed
   target range (e.g., `[0, π/2]` instead of `[0, π]`) to reduce the
   oscillatory frequency of the ZZ interactions.

2. **Kernel alignment:** Use `KernelAlignment` (Cortes et al., 2012) to select
   the `reps` and entanglement configuration whose kernel matrix is most aligned
   with the ideal label kernel $y_i y_j$.

3. **Hybrid kernel:** Combine the quantum kernel with the RBF kernel via a
   convex combination $K_{\text{hybrid}} = \alpha K_{\text{RBF}} + (1-\alpha) K_{\text{QNN}}$,
   leveraging the complementary inductive biases of both.

4. **Reduce to `reps=2` with partial entanglement:** The `linear` entanglement
   pattern provides only nearest-neighbour CX pairs (3 pairs for 4 qubits)
   and `reps=2` keeps the expressibility in the "sweet spot" — sufficient to
   capture non-linear class boundaries without incurring kernel degeneracy.

---

### Conclusion

The over-expressibility paradox is a known challenge in near-term quantum machine
learning (Thanasilp et al., *Nature Communications* 2022; Kübler et al., *NeurIPS*
2021). A more complex quantum circuit is not necessarily a better one — the
circuit's inductive bias must match the geometry of the target dataset.

For the NSL-KDD intrusion detection task with 4 PCA features, the `reps=3,
full` ZZFeatureMap is over-parameterised relative to the data manifold's
complexity. The classical RBF kernel's translation-invariant, locally-smooth
inductive bias provides a better fit for this specific dataset slice —
a result consistent with the **no free lunch theorem** in machine learning.